# TextMamba3D — A100 Training Pipeline (V5.1)

**V5.1: Real Mamba-3 Complex-Valued SSM (ICLR 2026)**

| Feature | V5.1a (control) | V5.1b (full) |
|---------|----------------|---------------|
| SSM Backend | Real Mamba-3 | Real Mamba-3 |
| d_state | 16 (match V5.0) | 64 (Mamba-3 sweet spot) |
| chunk_size | 64 | 128 |
| Precision | Pure bf16 | Pure bf16 |
| AMP | Disabled | Disabled |
| grad_checkpoint | Disabled | Disabled |

Config: `configs/textbrats_a100_v5.1.yaml` (notebook patches d_state/chunk_size for V5.1a)

> **Key change from V5.0:** Real Mamba-3 from GitHub source replaces Mamba-2 PyPI fallback.
> Complex-valued SSM with RoPE + exponential-trapezoidal discretization.
> Targets TC Dice regression via rotational dynamics across 3-axis cross-scan.

In [ ]:
# Cell 1: Install Mamba-3 from GitHub source + deps
from google.colab import drive
drive.mount('/content/drive')

!nvidia-smi 2>/dev/null || echo "No GPU"

# Step 1: Install PyPI mamba-ssm (provides compiled CUDA extensions)
!pip install -q --cache-dir=/content/drive/MyDrive/pip_cache \
    mamba-ssm causal-conv1d einops \
    transformers nibabel tensorboard pyyaml tqdm

# Step 2: Install Mamba-3 runtime deps
!pip install -q "triton>=3.5.0" "tilelang>=0.1.7.post3" "nvidia-cutlass-dsl==4.4.1" "quack-kernels==0.3.1" cuda-python

# Step 3: Clone Mamba-3 source (provides Python-level Mamba3 module)
import sys, os, shutil
SRC = "/content/mamba_src"
if os.path.isdir(SRC):
    shutil.rmtree(SRC)
!git clone --depth 1 https://github.com/state-spaces/mamba.git {SRC}

assert os.path.exists(f"{SRC}/mamba_ssm/modules/mamba3.py"), "mamba3.py not found"

# Step 4: Stub out CUTLASS step fn (not needed for training)
stub_path = f"{SRC}/mamba_ssm/ops/cute/mamba3/mamba3_step_fn.py"
if os.path.exists(stub_path):
    shutil.copy2(stub_path, stub_path + ".bak")
    with open(stub_path, "w") as f:
        f.write("def mamba3_step_fn(*args, **kwargs):\n    raise NotImplementedError('mamba3_step_fn requires CUTLASS DSL')\n")
    print("Patched: mamba3_step_fn stubbed")

# Step 5: Overlay source path (source Mamba3 module + PyPI compiled extensions)
for key in list(sys.modules.keys()):
    if "mamba_ssm" in key:
        del sys.modules[key]
if SRC not in sys.path:
    sys.path.insert(0, SRC)

# Verify
from mamba_ssm import Mamba, Mamba2, Mamba3
print(f"Mamba3: {Mamba3}")
print("Real Mamba-3 installed OK (source overlay on PyPI compiled extensions)")

In [ ]:
# Cell 2: Clone repo + extract data
import os, zipfile, shutil, subprocess, time

REPO_DIR = '/content/TextMamba3D'
DRIVE_BASE = '/content/drive/MyDrive/TextMamba3D'

git_dir = os.path.join(REPO_DIR, '.git')
if os.path.isdir(REPO_DIR) and not os.path.isdir(git_dir):
    print(f'Removing old non-git code at {REPO_DIR}...')
    shutil.rmtree(REPO_DIR)

if os.path.isdir(git_dir):
    os.chdir(REPO_DIR)
    subprocess.run(['git', 'pull'], check=True)
    print(f'Updated existing repo at {REPO_DIR}')
else:
    for attempt in range(1, 4):
        print(f'Cloning (attempt {attempt}/3)...')
        ret = subprocess.run(
            ['git', 'clone', '--depth', '1', 'https://github.com/PlutoLei/TextMamba3D.git', REPO_DIR],
            capture_output=True, text=True
        )
        if ret.returncode == 0 and os.path.exists(os.path.join(REPO_DIR, 'models/textmamba3d.py')):
            break
        print(f'  Failed (code {ret.returncode}): {ret.stderr.strip()}')
        if os.path.isdir(REPO_DIR):
            shutil.rmtree(REPO_DIR)
        if attempt < 3:
            time.sleep(5 * attempt)
    else:
        raise RuntimeError(f'Clone failed after 3 attempts. Last error: {ret.stderr.strip()}')
    os.chdir(REPO_DIR)
    print(f'Cloned to {REPO_DIR}')

print(f'Working directory: {os.getcwd()}')

DATA_ZIP = os.path.join(DRIVE_BASE, "TextBraTS_data.zip")
DATA_DIR = os.path.join(REPO_DIR, "data/BraTS2020/BraTS2020_TrainingData/MICCAI_BraTS2020_TrainingData")

if not os.path.exists(DATA_DIR):
    os.makedirs(os.path.dirname(DATA_DIR), exist_ok=True)
    if os.path.exists(DATA_ZIP):
        print(f"Extracting {DATA_ZIP}...")
        with zipfile.ZipFile(DATA_ZIP, 'r') as zf:
            zf.extractall(os.path.dirname(DATA_DIR))
        if os.path.exists(DATA_DIR):
            print(f"Data extracted. Cases: {len(os.listdir(DATA_DIR))}")
        else:
            print(f"ERROR: Expected path not found after extraction: {DATA_DIR}")
    else:
        print(f"ERROR: {DATA_ZIP} not found on Drive")
else:
    print(f"Data already exists. Cases: {len(os.listdir(DATA_DIR))}")

if os.path.exists(DATA_DIR):
    cases = [d for d in os.listdir(DATA_DIR) if os.path.isdir(os.path.join(DATA_DIR, d))]
    print(f"Total BraTS cases: {len(cases)}")

In [ ]:
# Cell 3: ET-enriched text restore
import os, sys, zipfile
os.chdir(REPO_DIR)

DATA_DIR = "./data/BraTS2020/BraTS2020_TrainingData/MICCAI_BraTS2020_TrainingData"
ET_CACHE_ZIP = os.path.join(DRIVE_BASE, "et_enriched.zip")

cases = sorted(
    d for d in os.listdir(DATA_DIR)
    if os.path.isdir(os.path.join(DATA_DIR, d))
) if os.path.isdir(DATA_DIR) else []
if not cases:
    raise RuntimeError(f"No BraTS cases found in {DATA_DIR}")

sample_enriched = os.path.join(DATA_DIR, cases[0], f"{cases[0]}_et_enriched.txt")

if os.path.exists(sample_enriched):
    count = sum(1 for d in cases if os.path.exists(os.path.join(DATA_DIR, d, f"{d}_et_enriched.txt")))
    print(f"ET-enriched text already present for {count} cases, skipping")
elif os.path.exists(ET_CACHE_ZIP):
    print(f"Restoring ET-enriched text from {ET_CACHE_ZIP}...")
    with zipfile.ZipFile(ET_CACHE_ZIP, 'r') as zf:
        zf.extractall(DATA_DIR)
    count = sum(1 for d in cases if os.path.exists(os.path.join(DATA_DIR, d, f"{d}_et_enriched.txt")))
    print(f"Restored ET-enriched text for {count} cases")
else:
    print("Generating ET-enriched text descriptions from T1ce images...")
    sys.path.insert(0, '.')
    from data.et_text_enrichment import process_all_cases
    results = process_all_cases(DATA_DIR)
    print(f"Generated for {len(results)} cases")
    with zipfile.ZipFile(ET_CACHE_ZIP, 'w', zipfile.ZIP_DEFLATED) as zf:
        for case_dir in cases:
            et_file = os.path.join(DATA_DIR, case_dir, f"{case_dir}_et_enriched.txt")
            if os.path.exists(et_file):
                zf.write(et_file, os.path.join(case_dir, f"{case_dir}_et_enriched.txt"))
    print(f"Cached ET text to {ET_CACHE_ZIP}")

et_count = sum(1 for d in cases if os.path.exists(os.path.join(DATA_DIR, d, f'{d}_et_enriched.txt')))
if et_count == 0:
    raise RuntimeError('No ET-enriched text files found. Run text generation first.')
print(f'ET-enriched text verified: {et_count}/{len(cases)} cases')

In [ ]:
# Cell 4: Verify real Mamba3 + pure bf16
import sys, os, torch, yaml, inspect
os.chdir(REPO_DIR)
sys.path.insert(0, '.')

from models.mamba_block import MAMBA3_AVAILABLE, MAMBA3_IS_REAL, _create_ssm, _auto_headdim
print(f"MAMBA3_AVAILABLE: {MAMBA3_AVAILABLE}")
print(f"MAMBA3_IS_REAL: {MAMBA3_IS_REAL}")
assert MAMBA3_IS_REAL, "Must use REAL Mamba-3, not Mamba-2 fallback!"

# Verify auto_headdim
for dim in [48, 96, 192, 384]:
    d_inner = dim * 2
    hd = _auto_headdim(d_inner)
    assert d_inner % hd == 0
    print(f"  Stage dim={dim} -> d_inner={d_inner} -> auto headdim={hd}")

# Verify V5.1 params in TextMamba3D
from models.textmamba3d import TextMamba3D
sig = inspect.signature(TextMamba3D.__init__)
for param in ['use_mamba3', 'headdim', 'rope_fraction', 'chunk_size', 'is_mimo']:
    assert param in sig.parameters, f"Missing V5.1 param: {param}!"
print("TextMamba3D: all V5.1 params present")

# Verify config
with open('configs/textbrats_a100_v5.1.yaml') as f:
    cfg = yaml.safe_load(f)
assert cfg['model']['use_mamba3'] is True
assert cfg['training']['use_amp'] is False
assert cfg['training']['bf16_mode'] == 'pure'
print(f"Config verified: use_mamba3=True, use_amp=False, bf16_mode=pure")

# Quick Mamba3 bf16 forward+backward test
device = torch.device('cuda')
from mamba_ssm import Mamba3
ssm = Mamba3(d_model=96, d_state=16, expand=2).to(device=device, dtype=torch.bfloat16)
x = torch.randn(1, 64, 96, device=device, dtype=torch.bfloat16, requires_grad=True)
out = ssm(x)
out.sum().backward()
assert not torch.isnan(x.grad).any()
print(f"Mamba3 bf16 forward+backward: OK")
del ssm, x, out
torch.cuda.empty_cache()
print("All V5.1 checks passed!")

In [ ]:
# Cell 5: Smoke test (pure bf16, 2 samples)
import os
os.chdir(REPO_DIR)

print("Running V5.1 smoke test (pure bf16)...")
!python -u train.py \
    --config configs/textbrats_a100_v5.1.yaml \
    --max-samples 2 \
    --max-epochs 1 \
    --no-text-ratio 0.0 \
    --grad-accum 1

import torch
if torch.cuda.is_available():
    peak = torch.cuda.max_memory_allocated() / 1024**3
    total = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"Peak GPU memory: {peak:.1f} / {total:.0f} GB")
    torch.cuda.reset_peak_memory_stats()

## V5.1a Training: Control Experiment (d_state=16)

Same d_state as V5.0 (Mamba-2) — isolates the effect of Real Mamba-3 kernel vs Mamba-2.

| Parameter | Value |
|-----------|-------|
| d_state | 16 |
| chunk_size | 64 |
| batch_size | 2, grad_accum=2 (effective 4) |
| Precision | Pure bf16 (no AMP) |

In [ ]:
# Cell 7: V5.1a Training
import subprocess, shutil, glob, os, yaml

os.chdir(REPO_DIR)
DRIVE_CKPT = os.path.join(DRIVE_BASE, "checkpoints")
os.makedirs(DRIVE_CKPT, exist_ok=True)

def sync_checkpoints_to_drive(tag):
    """Sync and tag checkpoints to Drive."""
    local_ckpt = os.path.join(REPO_DIR, "checkpoints")
    if not os.path.exists(local_ckpt):
        return
    for f in glob.glob(os.path.join(local_ckpt, "*.pth")):
        dst = os.path.join(DRIVE_CKPT, os.path.basename(f))
        shutil.copy2(f, dst)
    # Tag best checkpoint
    local_best = os.path.join(local_ckpt, "best.pth")
    if os.path.exists(local_best):
        tagged = os.path.join(DRIVE_CKPT, f"best_{tag}.pth")
        shutil.copy2(local_best, tagged)
        print(f"Tagged checkpoint: {tagged}")
    print(f"Synced checkpoints to {DRIVE_CKPT}")

# Clean local checkpoints
for f in glob.glob(os.path.join(REPO_DIR, "checkpoints/*.pth")):
    os.remove(f)
print("Cleaned local checkpoints for V5.1a fresh start")

# Patch config for V5.1a: d_state=16, chunk_size=64
with open('configs/textbrats_a100_v5.1.yaml') as f:
    cfg = yaml.safe_load(f)
cfg['model']['d_state'] = 16
cfg['model']['chunk_size'] = 64
cfg['experiment']['name'] = 'TextMamba3D_A100_v5.1a_real_mamba3'
with open('configs/_v51a_runtime.yaml', 'w') as f:
    yaml.dump(cfg, f, default_flow_style=False)
print("V5.1a runtime config: d_state=16, chunk_size=64")

# Train V5.1a
ret = subprocess.run(
    ['python', '-u', 'train.py',
     '--config', 'configs/_v51a_runtime.yaml',
     '--no-text-ratio', '0.15',
     '--grad-accum', '2'],
    cwd=REPO_DIR,
)
if ret.returncode != 0:
    raise RuntimeError(f"V5.1a training failed with exit code {ret.returncode}")

sync_checkpoints_to_drive("v5.1a")

# Archive V5.1a logs to Drive before V5.1b overwrites them
local_logs = os.path.join(REPO_DIR, "logs")
if os.path.isdir(local_logs):
    drive_logs_a = os.path.join(DRIVE_BASE, "logs_v5.1a")
    if os.path.isdir(drive_logs_a):
        shutil.rmtree(drive_logs_a)
    shutil.copytree(local_logs, drive_logs_a)
    print(f"Archived V5.1a logs to {drive_logs_a}")

print("V5.1a training complete!")

In [ ]:
# Cell 8: V5.1a 8-config Eval
import subprocess, os
os.chdir(REPO_DIR)

ckpt = os.path.join(DRIVE_CKPT, "best_v5.1a.pth")
if not os.path.exists(ckpt):
    ckpt = os.path.join(REPO_DIR, "checkpoints/best.pth")
assert os.path.exists(ckpt), f"No V5.1a checkpoint found at {ckpt}"

CONFIG = 'configs/_v51a_runtime.yaml'
results_v51a = {}

eval_configs = [
    ("text",          ["--use-text"]),
    ("text+PP",       ["--use-text", "--postprocess"]),
    ("text+TTA",      ["--use-text", "--tta"]),
    ("text+TTA+PP",   ["--use-text", "--tta", "--postprocess"]),
    ("notext",        ["--no-text"]),
    ("notext+PP",     ["--no-text", "--postprocess"]),
    ("notext+TTA",    ["--no-text", "--tta"]),
    ("notext+TTA+PP", ["--no-text", "--tta", "--postprocess"]),
]

print("=" * 70)
print("V5.1a EVALUATION (d_state=16, Real Mamba-3)")
print("=" * 70)

for name, flags in eval_configs:
    print(f"\n--- {name} ---")
    cmd = [
        'python', '-u', 'evaluate_full.py',
        '--config', CONFIG,
        '--checkpoint', ckpt,
        '--split', 'test',
        '--overlap', '0.5',
    ] + flags
    ret = subprocess.run(cmd, cwd=REPO_DIR, capture_output=True, text=True)
    print(ret.stdout[-500:] if len(ret.stdout) > 500 else ret.stdout)
    if ret.returncode != 0:
        print(f"WARNING: {name} eval failed: {ret.stderr[-300:]}")

print("\nV5.1a evaluation complete!")
print("Copy the Dice scores from above into the comparison cell.")

## V5.1b Training: Full Mamba-3 Potential (d_state=64)

d_state=64 matches Mamba-3 paper's recommended configuration for optimal tensor core utilization.
chunk_size=128 reduces kernel launches.

| Parameter | Value |
|-----------|-------|
| d_state | 64 |
| chunk_size | 128 |
| batch_size | 2, grad_accum=2 (effective 4) |
| Precision | Pure bf16 (no AMP) |

In [ ]:
# Cell 10: V5.1b Training
import subprocess, shutil, glob, os
os.chdir(REPO_DIR)

# Clean local checkpoints from V5.1a
for f in glob.glob(os.path.join(REPO_DIR, "checkpoints/*.pth")):
    os.remove(f)
print("Cleaned local checkpoints for V5.1b fresh start")

# V5.1b uses the default config (d_state=64, chunk_size=128)
ret = subprocess.run(
    ['python', '-u', 'train.py',
     '--config', 'configs/textbrats_a100_v5.1.yaml',
     '--no-text-ratio', '0.15',
     '--grad-accum', '2'],
    cwd=REPO_DIR,
)
if ret.returncode != 0:
    raise RuntimeError(f"V5.1b training failed with exit code {ret.returncode}")

sync_checkpoints_to_drive("v5.1b")

# Archive V5.1b logs to Drive
local_logs = os.path.join(REPO_DIR, "logs")
if os.path.isdir(local_logs):
    drive_logs_b = os.path.join(DRIVE_BASE, "logs_v5.1b")
    if os.path.isdir(drive_logs_b):
        shutil.rmtree(drive_logs_b)
    shutil.copytree(local_logs, drive_logs_b)
    print(f"Archived V5.1b logs to {drive_logs_b}")

print("V5.1b training complete!")

In [ ]:
# Cell 11: V5.1b 8-config Eval
import subprocess, os
os.chdir(REPO_DIR)

ckpt = os.path.join(DRIVE_CKPT, "best_v5.1b.pth")
if not os.path.exists(ckpt):
    ckpt = os.path.join(REPO_DIR, "checkpoints/best.pth")
assert os.path.exists(ckpt), f"No V5.1b checkpoint found at {ckpt}"

CONFIG = 'configs/textbrats_a100_v5.1.yaml'

print("=" * 70)
print("V5.1b EVALUATION (d_state=64, Real Mamba-3)")
print("=" * 70)

for name, flags in eval_configs:
    print(f"\n--- {name} ---")
    cmd = [
        'python', '-u', 'evaluate_full.py',
        '--config', CONFIG,
        '--checkpoint', ckpt,
        '--split', 'test',
        '--overlap', '0.5',
    ] + flags
    ret = subprocess.run(cmd, cwd=REPO_DIR, capture_output=True, text=True)
    print(ret.stdout[-500:] if len(ret.stdout) > 500 else ret.stdout)
    if ret.returncode != 0:
        print(f"WARNING: {name} eval failed: {ret.stderr[-300:]}")

print("\nV5.1b evaluation complete!")

## Results Comparison

Fill in Dice scores from V5.1a and V5.1b eval outputs above.
Compare with V5.0 baseline (Mamba-2, d_state=16):
- V5.0 text+TTA+PP: ET=0.7910, TC=0.8560, WT=0.8967, Mean=0.8479

In [ ]:
# Cell 13: Comparison table + visualization
import matplotlib.pyplot as plt
import numpy as np

# V5.0 baseline (Mamba-2, d_state=16)
v50 = {'ET': 0.7910, 'TC': 0.8560, 'WT': 0.8967, 'Mean': 0.8479}

# Fill in after eval (text+TTA+PP results)
v51a = {'ET': 0.0, 'TC': 0.0, 'WT': 0.0, 'Mean': 0.0}  # d_state=16
v51b = {'ET': 0.0, 'TC': 0.0, 'WT': 0.0, 'Mean': 0.0}  # d_state=64

if v51a['Mean'] == 0.0 or v51b['Mean'] == 0.0:
    print("Fill in V5.1a and V5.1b results from eval cells above, then re-run.")
    print()
    print("Key questions:")
    print("  1. V5.1a vs V5.0: Real Mamba-3 vs Mamba-2 (same d_state=16)")
    print("  2. V5.1b vs V5.1a: d_state=64 vs d_state=16 (same Mamba-3)")
    print("  3. V5.1b vs V5.0: Full Mamba-3 vs Mamba-2 baseline")
else:
    labels = list(v50.keys())
    x = np.arange(len(labels))
    w = 0.25

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

    ax1.bar(x - w, [v50[k] for k in labels], w, label='V5.0 (Mamba-2, d=16)', color='steelblue', alpha=0.8)
    ax1.bar(x, [v51a[k] for k in labels], w, label='V5.1a (Mamba-3, d=16)', color='coral', alpha=0.8)
    ax1.bar(x + w, [v51b[k] for k in labels], w, label='V5.1b (Mamba-3, d=64)', color='forestgreen', alpha=0.8)
    ax1.set_ylabel('Dice Score')
    ax1.set_title('V5.0 vs V5.1a vs V5.1b (text+TTA+PP)')
    ax1.set_xticks(x)
    ax1.set_xticklabels(labels)
    ax1.legend()
    ax1.grid(axis='y', alpha=0.3)
    ax1.set_ylim(0.75, 0.95)

    # Delta chart
    d_51a = [v51a[k] - v50[k] for k in labels]
    d_51b = [v51b[k] - v50[k] for k in labels]
    ax2.bar(x - w/2, d_51a, w, label='V5.1a - V5.0', color='coral', alpha=0.8)
    ax2.bar(x + w/2, d_51b, w, label='V5.1b - V5.0', color='forestgreen', alpha=0.8)
    ax2.axhline(y=0, color='black', linewidth=0.5)
    ax2.set_ylabel('Delta')
    ax2.set_title('Improvement over V5.0')
    ax2.set_xticks(x)
    ax2.set_xticklabels(labels)
    ax2.legend()
    ax2.grid(axis='y', alpha=0.3)

    plt.tight_layout()
    plt.savefig('v51_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()

    print("Summary:")
    print(f"  V5.0  best: Mean={v50['Mean']:.4f}")
    print(f"  V5.1a best: Mean={v51a['Mean']:.4f} (delta={v51a['Mean']-v50['Mean']:+.4f})")
    print(f"  V5.1b best: Mean={v51b['Mean']:.4f} (delta={v51b['Mean']-v50['Mean']:+.4f})")

## Resume Training (After Disconnect)

To resume V5.1a or V5.1b after a disconnect, use the cell below.
Change the config and tag as needed.

In [ ]:
# Cell 15: Resume cell
import os, shutil
os.chdir(REPO_DIR)

# CHANGE THESE for V5.1a vs V5.1b
RESUME_CONFIG = 'configs/textbrats_a100_v5.1.yaml'  # or '_v51a_runtime.yaml' for V5.1a
RESUME_TAG = 'v5.1b'  # or 'v5.1a'

resume_ckpt = os.path.join(DRIVE_CKPT, "last.pth")
if os.path.exists(resume_ckpt):
    print(f"Resuming {RESUME_TAG} from {resume_ckpt}")
    !python -u train.py \
        --config {RESUME_CONFIG} \
        --resume "{resume_ckpt}" \
        --no-text-ratio 0.15 \
        --grad-accum 2

    sync_checkpoints_to_drive(RESUME_TAG)
else:
    print(f"No checkpoint to resume from at {resume_ckpt}")